# Time-Series Analysis with Exponential Fitting

This notebook performs time-series analysis of spectral data at a specific wavelength using exponential decay fitting.

## Overview
- Load processed spectral data from CSV
- Extract absorbance at a specific wavelength over time
- Fit exponential decay model
- Visualize results and extract decay parameters

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lmfit.models import ExponentialModel

# Enable inline plotting
%matplotlib inline

## Load Data

Update the file path below to point to your processed spectral data:

In [ ]:
# Update this path to your data file
data_file = 'C:/Users/jake_/Desktop/HGD_nanospec/DTNB_5/hgd_R37S_DTNB_1_5_transmission.asc_pyspec/final_pyspec.csv'

# Load the data
data = pd.read_csv(data_file, index_col=0)
print(f"Data shape: {data.shape}")
print("\nFirst few rows:")
display(data.head())

## Extract Time and Wavelength Information

In [ ]:
# Strip units and convert to float
Time = data.columns.str.replace('s', '').astype(float)
wavelengths = data.index.values.astype(float)

print(f"Time range: {Time.min():.2f} - {Time.max():.2f} s")
print(f"Wavelength range: {wavelengths.min():.1f} - {wavelengths.max():.1f} nm")
print(f"Number of time points: {len(Time)}")
print(f"Number of wavelengths: {len(wavelengths)}")

## Configuration

Set the wavelength and time cutoff for analysis:

In [ ]:
# Specify the wavelength and time point you want to analyze
selected_wavelength = 412  # nm
cut_timepoint = 20  # seconds

print(f"Selected wavelength: {selected_wavelength} nm")
print(f"Time cutoff: {cut_timepoint} s")

## Find Closest Wavelength

The data may not have the exact wavelength requested, so find the closest available:

In [ ]:
# Function to find the closest wavelength
def find_closest_wavelength(selected_wavelength, available_wavelengths):
    return min(available_wavelengths, key=lambda x: abs(x - selected_wavelength))

# Find the closest recorded wavelength
closest_wavelength = find_closest_wavelength(selected_wavelength, wavelengths)
print(f"Requested wavelength: {selected_wavelength} nm")
print(f"Closest available wavelength: {closest_wavelength} nm")

## Extract and Filter Data

In [ ]:
# Check if the closest wavelength is in the data
if closest_wavelength in wavelengths:
    Absorbance = data.loc[closest_wavelength].values
    
    # Cut the data at the specified time point
    mask = Time <= cut_timepoint
    Time_cut = Time[mask]
    Absorbance_cut = Absorbance[mask]
    
    print(f"Data points before cutoff: {len(Time)}")
    print(f"Data points after cutoff: {len(Time_cut)}")
    print(f"Absorbance range: {Absorbance_cut.min():.4f} - {Absorbance_cut.max():.4f}")
else:
    print(f'Wavelength {closest_wavelength} not found in the data.')
    raise ValueError("Wavelength not found")

## Visualize Raw Data

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(Time_cut, Absorbance_cut, s=20, alpha=0.6)
plt.xlabel('Time (s)')
plt.ylabel('Absorbance')
plt.title(f'Raw Data at {closest_wavelength} nm')
plt.grid(True, alpha=0.3)
plt.show()

## Define Weights

Reduce the influence of noisy data points (e.g., early time points):

In [ ]:
# Define weights to reduce the influence of noisy data points
weights = np.ones_like(Absorbance_cut)
weights[Time_cut < 2] = 0  # Assign zero weights to data points before 2 seconds

print(f"Number of points with weight = 0: {np.sum(weights == 0)}")
print(f"Number of points with weight = 1: {np.sum(weights == 1)}")

## Fit Exponential Model

In [ ]:
# Create Exponential model
exp_mod = ExponentialModel(prefix='exp_')

# Create parameters for the model
params = exp_mod.make_params(amplitude=0.1, decay=5)
print("Initial parameters:")
print(params)

# Fit the model to the data with weights
out = exp_mod.fit(Absorbance_cut, params, x=Time_cut, weights=weights)

print("\n" + "="*60)
print("Fit completed successfully!")
print("="*60)

## View Fit Report

In [ ]:
print(out.fit_report(min_correl=0.3))

## Visualize Fit Results

In [ ]:
# Plot the data and the fit
plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.scatter(Time_cut, Absorbance_cut, label='Data', s=20, alpha=0.6)
plt.plot(Time_cut, out.init_fit, label='Initial fit', linestyle='--', linewidth=2)
plt.plot(Time_cut, out.best_fit, label='Best fit', linewidth=2)

# Plot the individual model component
components = out.eval_components(x=Time_cut)
plt.plot(Time_cut, components['exp_'], label='Exponential component', linestyle=':', linewidth=2)

plt.legend()
plt.title(f'Fit for wavelength {closest_wavelength} nm')
plt.xlabel('Time (s)')
plt.ylabel('Absorbance')
plt.grid(True, alpha=0.3)

# Second plot: simpler view
plt.subplot(2, 1, 2)
plt.scatter(Time_cut, Absorbance_cut, s=20, alpha=0.6)
plt.plot(Time_cut, out.best_fit, linewidth=2, color='red')
plt.xlabel('Time (s)')
plt.ylabel('Absorbance')
plt.title(f'Data and Best Fit at {closest_wavelength} nm')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Extract Fit Parameters

In [ ]:
# Extract key parameters
amplitude = out.params['exp_amplitude'].value
decay = out.params['exp_decay'].value

print(f"\nFitted Parameters:")
print(f"  Amplitude: {amplitude:.6f}")
print(f"  Decay constant: {decay:.6f} s")
print(f"  Half-life (t½): {decay * np.log(2):.6f} s")
print(f"  R-squared: {out.rsquared:.6f}")
print(f"  Reduced chi-square: {out.redchi:.6e}")

## Save Fit Report

Save the fit report to a log file:

In [ ]:
# Write the fit report to a log file
with open('time_analysis_fit_report.log', 'w') as f:
    f.write(f'Spectra wavelength: {closest_wavelength}\n')
    f.write(out.fit_report(min_correl=0.3))

print(f"Fit report saved to 'time_analysis_fit_report.log'")

## Save Plots

In [ ]:
# Save plot with legend
plt.figure(figsize=(10, 6))
plt.scatter(Time_cut, Absorbance_cut, label='Data', s=20, alpha=0.6)
plt.plot(Time_cut, out.init_fit, label='Initial fit', linestyle='--', linewidth=2)
plt.plot(Time_cut, out.best_fit, label='Best fit', linewidth=2)
components = out.eval_components(x=Time_cut)
plt.plot(Time_cut, components['exp_'], label='Exponential component', linestyle=':', linewidth=2)
plt.legend()
plt.title(f'Fit for wavelength {closest_wavelength} nm')
plt.xlabel('Time (s)')
plt.ylabel('Absorbance')
plt.grid(True, alpha=0.3)
plt.savefig(f'fit_{closest_wavelength}_nm.png', dpi=300, bbox_inches='tight')
plt.close()

# Save plot without legend
plt.figure(figsize=(10, 6))
plt.scatter(Time_cut, Absorbance_cut, s=20, alpha=0.6)
plt.plot(Time_cut, out.best_fit, linewidth=2, color='red')
plt.xlabel('Time (s)')
plt.ylabel('Absorbance')
plt.title(f'Fit for wavelength {closest_wavelength} nm')
plt.grid(True, alpha=0.3)
plt.savefig(f'fit_{closest_wavelength}_nm_no_legend.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"Plots saved:")
print(f"  - fit_{closest_wavelength}_nm.png")
print(f"  - fit_{closest_wavelength}_nm_no_legend.png")